# 10 · Changing data: INSERT, UPDATE, MERGE, SCD Type 2, incremental loads (Oracle)

**DML** (Data Manipulation Language) changes rows: `INSERT`, `UPDATE`, `DELETE`, `MERGE`.
**DDL** (Data Definition Language) changes tables: `CREATE`, `ALTER`, `DROP`.

This notebook works on **copies** named `prac_*`, so the 7 practice tables never change.
Each question has a setup cell that rebuilds its copy: run it first, and again to start over.

Data engineering words used here:
- **upsert** = update the row if the key exists, insert it if not (`MERGE`)
- **SCD Type 2** (slowly changing dimension) = keep the **history**: close the old row, add a new current row
- **incremental load** = load only the new rows since last time (using a **watermark**, e.g. the max date already loaded)
- **idempotent** = running the job twice gives the same result as running it once

**How to run:** Start each SQL cell with `%%oracle`. One statement per cell. Tables are in your own schema, so just write `employees`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`oracle/practice/10_dml_merge_scd.ipynb`](../../oracle/practice/10_dml_merge_scd.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "oracle"))
%load_ext oracle_helper

## Q1. INSERT a row  ·  🟢 easy

Add customer **41**, `Test User`, city `Austin`, signup `2024-12-01` to `prac_customers`. Then show `num_customers` (the row count).

*Expected result: 1 row · columns: `num_customers`*

**How it works:** `INSERT INTO table (columns) VALUES (...)` adds one row; the SELECT COUNT checks it is there.

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

OK, 40 row(s) affected

In [ ]:
%%oracle
INSERT INTO prac_customers (customer_id, name, city, signup_date)
VALUES (41, 'Test User', 'Austin', DATE '2024-12-01')

OK, 1 row(s) affected

In [ ]:
%%oracle
SELECT COUNT(*) AS num_customers FROM prac_customers

,num_customers
0,41


**Why / traps:** Always list the columns in INSERT. If someone adds a column later, your insert still works.

The notebook helper commits after each statement. In SQL*Plus or SQL Developer you must run `COMMIT;` yourself (or `ROLLBACK;` to undo).

## Q2. UPDATE with a condition  ·  🟢 easy

In `prac_customers`, set `city` to `Unknown` where it is NULL. Then show `customer_id`, `name`, `city` of the changed rows.

*Expected result: 1 row · columns: `customer_id`, `name`, `city`*

**How it works:** `UPDATE ... SET city = 'Unknown' WHERE city IS NULL` changes only the matching rows.

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

OK, 40 row(s) affected

In [ ]:
%%oracle
UPDATE prac_customers SET city = 'Unknown' WHERE city IS NULL

OK, 1 row(s) affected

In [ ]:
%%oracle
SELECT customer_id, name, city FROM prac_customers WHERE city = 'Unknown'

,customer_id,name,city
0,13,Lucas Lopez,Unknown


**Why / traps:** Write the WHERE first and test it as a SELECT. An UPDATE without WHERE changes **every** row.

## Q3. DELETE with NOT EXISTS  ·  🟢 easy

Delete from `prac_customers` everyone who **never ordered**. Then show `num_customers`.

*Expected result: 1 row · columns: `num_customers`*

**How it works:** `DELETE ... WHERE NOT EXISTS (order for this customer)` removes customers who never ordered.

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

OK, 40 row(s) affected

In [ ]:
%%oracle
DELETE FROM prac_customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)

OK, 8 row(s) affected

In [ ]:
%%oracle
SELECT COUNT(*) AS num_customers FROM prac_customers

,num_customers
0,32


**Why / traps:** `TRUNCATE TABLE` removes **all** rows fast (no WHERE). In Oracle it cannot be rolled back.

## Q4. Upsert with MERGE  ·  🟡 medium

`prac_customer_updates` has 4 rows: customers 1 and 2 changed, 41 and 42 are new. MERGE them into `prac_customers`: update `name` and `city` when the id exists, insert when not. Then show customers 1, 2, 41, 42.

*Expected result: 4 rows · columns: `customer_id`, `name`, `city`, `signup_date`*

**How it works:** MERGE matches source to target on the key: `WHEN MATCHED` → UPDATE the row, `WHEN NOT MATCHED` → INSERT it. One statement does both.

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

OK, 40 row(s) affected

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customer_updates PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_customer_updates (
    customer_id NUMBER(10), name VARCHAR2(100), city VARCHAR2(50), signup_date DATE)

OK

In [ ]:
%%oracle
INSERT INTO prac_customer_updates
SELECT 1,  'Aarav Nguyen',    'Denver',  DATE '2023-01-26' FROM dual UNION ALL
SELECT 2,  'Priya Silva-Rao', 'Seattle', DATE '2023-09-08' FROM dual UNION ALL
SELECT 41, 'Nina Patel',      'Boston',  DATE '2024-12-05' FROM dual UNION ALL
SELECT 42, 'Omar Haddad',     'Austin',  DATE '2024-12-06' FROM dual

OK, 4 row(s) affected

In [ ]:
%%oracle
MERGE INTO prac_customers t
USING prac_customer_updates s
ON (t.customer_id = s.customer_id)
WHEN MATCHED THEN
  UPDATE SET t.name = s.name, t.city = s.city
WHEN NOT MATCHED THEN
  INSERT (customer_id, name, city, signup_date)
  VALUES (s.customer_id, s.name, s.city, s.signup_date)

OK, 4 row(s) affected

In [ ]:
%%oracle
SELECT * FROM prac_customers
WHERE customer_id IN (1, 2, 41, 42)
ORDER BY customer_id

,customer_id,name,city,signup_date
0,1,Aarav Nguyen,Denver,2023-01-26
1,2,Priya Silva-Rao,Seattle,2023-09-08
2,41,Nina Patel,Boston,2024-12-05
3,42,Omar Haddad,Austin,2024-12-06


**Why / traps:** MERGE fails (or does random things) if the **source has 2 rows for one key**. Dedupe the source first. MERGE is the main tool for upserts in data warehouses.

Oracle needs brackets around the ON condition: `ON (t.id = s.id)`.

## Q5. Idempotent reload of one slice  ·  🟡 medium

`prac_orders` is a raw copy of `orders` (with duplicates). Reload **October 2025** from a clean source (`SELECT DISTINCT * FROM orders`): first DELETE the slice, then INSERT it again. Run both cells twice: the result must not change. Show `oct_rows` and `oct_orders` for October 2025.

*Expected result: 1 row · columns: `oct_rows`, `oct_orders`*

**How it works:** Delete the whole October slice, then insert the clean October rows. Running both again gives the same rows (idempotent).

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_orders PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_orders AS
SELECT * FROM orders

OK, 108 row(s) affected

In [ ]:
%%oracle
DELETE FROM prac_orders
WHERE order_date >= DATE '2025-10-01' AND order_date < DATE '2025-11-01'

OK, 5 row(s) affected

In [ ]:
%%oracle
INSERT INTO prac_orders (order_id, customer_id, order_date, amount, status)
SELECT DISTINCT order_id, customer_id, order_date, amount, status
FROM orders
WHERE order_date >= DATE '2025-10-01' AND order_date < DATE '2025-11-01'

OK, 4 row(s) affected

In [ ]:
%%oracle
SELECT COUNT(*) AS oct_rows, COUNT(DISTINCT order_id) AS oct_orders
FROM prac_orders
WHERE order_date >= DATE '2025-10-01' AND order_date < DATE '2025-11-01'

,oct_rows,oct_orders
0,4,4


**Why / traps:** **Delete-then-insert by slice** (date, load id, ...) makes a job **idempotent**: a rerun replaces the slice instead of adding it twice. A plain INSERT on rerun = duplicates.

In Oracle, run DELETE + INSERT in **one transaction** (commit once at the end), so readers never see the slice half-loaded.

## Q6. Incremental load with a watermark  ·  🟡 medium

`prac_orders_target` holds clean orders up to 2025-06-30. Load only orders **newer than the latest date already in the target** (the watermark). Then show `num_rows` and `max_date` of the target.

*Expected result: 1 row · columns: `num_rows`, `max_date`*

**How it works:** The watermark = `MAX(order_date)` already in the target. Insert only source rows newer than it.

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_orders_target PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_orders_target AS
SELECT DISTINCT * FROM orders WHERE order_date <= DATE '2025-06-30'

OK, 74 row(s) affected

In [ ]:
%%oracle
INSERT INTO prac_orders_target (order_id, customer_id, order_date, amount, status)
SELECT DISTINCT order_id, customer_id, order_date, amount, status
FROM orders
WHERE order_date > (SELECT MAX(order_date) FROM prac_orders_target)

OK, 30 row(s) affected

In [ ]:
%%oracle
SELECT COUNT(*) AS num_rows, MAX(order_date) AS max_date FROM prac_orders_target

,num_rows,max_date
0,104,2025-12-28


**Why / traps:** Weak spot: a row that arrives **late** with an old date is never loaded. Real pipelines use a load timestamp / load id as the watermark, or reload the last N days (idempotent).

## Q7. SCD Type 2: keep the history  ·  🔴 hard

`prac_dim_customer` has one current row per customer (`valid_to = 9999-12-31`, `is_current = 'Y'`). `prac_city_changes` says: customer 1 → Denver, customer 3 → Phoenix (no change!), customer 13 → Austin (old city is NULL). Changes start on **2025-01-15**.

1. **Close** the current row of customers whose city really changed: `valid_to = 2025-01-14`, `is_current = 'N'`.
2. **Insert** a new current row for them from 2025-01-15.
3. Show the rows of customers 1, 3, 13.

*Expected result: 5 rows · columns: `customer_id`, `city`, `valid_from`, `valid_to`, `is_current`*

**How it works:** Step 1 closes the current row of customers whose city really changed (NULL-safe compare). Step 2 inserts a new current row for every change that now has no current row.

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_dim_customer PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_dim_customer AS
SELECT customer_id, city, signup_date AS valid_from,
       DATE '9999-12-31' AS valid_to, CAST('Y' AS VARCHAR2(1)) AS is_current
FROM customers

OK, 40 row(s) affected

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_city_changes PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_city_changes (customer_id NUMBER(10), city VARCHAR2(50))

OK

In [ ]:
%%oracle
INSERT INTO prac_city_changes
SELECT 1, 'Denver' FROM dual UNION ALL
SELECT 3, 'Phoenix' FROM dual UNION ALL      -- same city as now: NOT a change
SELECT 13, 'Austin' FROM dual                -- old city is NULL

OK, 3 row(s) affected

In [ ]:
%%oracle
UPDATE prac_dim_customer d
SET valid_to = DATE '2025-01-14',
    is_current = 'N'
WHERE d.is_current = 'Y'
  AND EXISTS (SELECT 1
              FROM prac_city_changes s
              WHERE s.customer_id = d.customer_id
                AND COALESCE(s.city, '~') <> COALESCE(d.city, '~'))

OK, 2 row(s) affected

In [ ]:
%%oracle
INSERT INTO prac_dim_customer (customer_id, city, valid_from, valid_to, is_current)
SELECT s.customer_id, s.city, DATE '2025-01-15', DATE '9999-12-31', 'Y'
FROM prac_city_changes s
WHERE NOT EXISTS (SELECT 1
                  FROM prac_dim_customer d
                  WHERE d.customer_id = s.customer_id
                    AND d.is_current = 'Y')

OK, 2 row(s) affected

In [ ]:
%%oracle
SELECT customer_id, city, valid_from, valid_to, is_current
FROM prac_dim_customer
WHERE customer_id IN (1, 3, 13)
ORDER BY customer_id, valid_from

,customer_id,city,valid_from,valid_to,is_current
0,1,Chicago,2023-01-26,2025-01-14 00:00:00,N
1,1,Denver,2025-01-15,9999-12-31 00:00:00,Y
2,3,Phoenix,2023-05-23,9999-12-31 00:00:00,Y
3,13,None,2023-10-12,2025-01-14 00:00:00,N
4,13,Austin,2025-01-15,9999-12-31 00:00:00,Y


**Why / traps:** NULL trap: `s.city <> d.city` is **unknown** when the old city is NULL, so customer 13 would be missed. COALESCE both sides (or a null-safe compare). Customer 3 must **not** get a new row. Order matters: close first, then insert the rows that have no current version.

## Q8. ALTER TABLE: add a flag column  ·  🟡 medium

Add a column `is_active` (1 letter) to `prac_customers`. Set it to `Y` for customers with at least one order, `N` for the rest. Show `is_active` and `num_customers`.

*Expected result: 2 rows · columns: `is_active`, `num_customers`*

**How it works:** `ALTER TABLE ... ADD` creates the empty column; one UPDATE fills it with CASE + EXISTS; GROUP BY checks the result.

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

OK, 40 row(s) affected

In [ ]:
%%oracle
ALTER TABLE prac_customers ADD (is_active VARCHAR2(1))

OK

In [ ]:
%%oracle
UPDATE prac_customers c
SET is_active = CASE WHEN EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
                     THEN 'Y' ELSE 'N' END

OK, 40 row(s) affected

In [ ]:
%%oracle
SELECT is_active, COUNT(*) AS num_customers
FROM prac_customers
GROUP BY is_active
ORDER BY is_active

,is_active,num_customers
0,N,8
1,Y,32


**Why / traps:** This is a **soft delete** style flag: keep the row, mark it inactive.